# 03 latent traversals

This notebook is an anonymized, output-stripped copy prepared for the GitHub study export. Paths are repo-relative; rerun cells after placing the required data and checkpoints.


# AR Step 4/6 Latent Traversal

Generalizes the `train-AR.ipynb` cLogP ridge-direction traversal over a concise property panel.
Very size- or sequence-confounded properties are excluded from primary traversal claims.


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

def find_project_root(start=None):
    p = (Path(start) if start is not None else Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError(f"Couldn't find project root from {p}")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from artifacts.model_compare.ar_model_h256_l256.patched_notebooks import ar_common as arc

ARTIFACT_ROOT = arc.ARTIFACT_ROOT
print("artifact_root =", ARTIFACT_ROOT)
for warning_text in arc.dependency_report():
    warnings.warn(warning_text)


In [ ]:
OUTPUT_DIR = ARTIFACT_ROOT / "traversal_step4_6"
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
for path in [OUTPUT_DIR, TABLES_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

STEP3_DIR = ARTIFACT_ROOT / "confounds_step3"
STEP3_TABLES = STEP3_DIR / "tables"


## Load Dataset, Checkpoint, And Step 3 Panel


In [ ]:
bundle = arc.load_dataset_and_splits()
model, device, ckpt_path = arc.load_ar_model(bundle)

panel_path = STEP3_DIR / "panels_step3_with_residuals.csv"
z_path = STEP3_DIR / "Z_mu.npy"
if not panel_path.exists() or not z_path.exists():
    raise FileNotFoundError("Run ar_step3_confounds.ipynb first so traversal uses the same Z, properties, and split.")

panel = pd.read_csv(panel_path)
Z = np.load(z_path).astype(np.float32)
split = panel["split"].to_numpy()

r2_c_path = STEP3_TABLES / "r2_C_to_Y.csv"
r2_resid_path = STEP3_TABLES / "r2_Z_to_Y_vs_Yresid.csv"
r2_C_to_Y = pd.read_csv(r2_c_path) if r2_c_path.exists() else pd.DataFrame(columns=["property", "r2_test"])
r2_resid = pd.read_csv(r2_resid_path) if r2_resid_path.exists() else pd.DataFrame(columns=["property", "r2_residual_test"])

print("checkpoint =", ckpt_path)
print("Z shape =", Z.shape)


## Select Traversal Panel


In [ ]:
VERY_CONFOUNDED_R2 = 0.85
manual_excluded = set(arc.EXCLUDED_CONFOUNDED_PROPERTIES)
high_confound = set(r2_C_to_Y.loc[r2_C_to_Y["r2_test"] >= VERY_CONFOUNDED_R2, "property"].tolist())
excluded_as_confounded = sorted(manual_excluded | high_confound)

available = [p for p in arc.TRAVERSAL_PROPERTIES if p in panel.columns and panel[p].notna().sum() > 100]
selected_properties = [p for p in available if p not in excluded_as_confounded]
controls = [p for p in available if p in excluded_as_confounded]

selection = pd.DataFrame({
    "property": available,
    "selected_primary": [p in selected_properties for p in available],
    "excluded_as_confounded_or_control": [p in excluded_as_confounded for p in available],
}).merge(r2_C_to_Y[["property", "r2_test"]].rename(columns={"r2_test": "confound_r2_test"}), on="property", how="left")
selection = selection.merge(r2_resid[["property", "r2_residual_test"]], on="property", how="left")
selection.to_csv(TABLES_DIR / "traversal_property_selection.csv", index=False)

print("selected traversal properties =", selected_properties)
print("excluded/control properties =", controls)
display(selection)


## Fit Ridge Directions And Traverse Test Seeds


In [ ]:
torch = arc.torch
ALPHAS = np.linspace(-20.0, 20.0, 21)
N_SEEDS_PER_PROPERTY = 3
rng = np.random.default_rng(arc.SEED)

def decoded_confound_values(selfies):
    try:
        toks = list(arc.sf.split_selfies(str(selfies))) if pd.notna(selfies) else []
    except Exception:
        toks = []
    return {
        "decoded_selfies_len_tokens": len(toks),
        "decoded_branch_token_count": sum("Branch" in tok for tok in toks),
        "decoded_ring_token_count": sum("Ring" in tok for tok in toks),
        "decoded_token_entropy": arc.shannon_entropy(toks),
    }

def actual_property(smiles, prop):
    props, _ = arc.build_property_panel([smiles], include_sa=(prop == "SA-score"))
    return float(props.iloc[0][prop]) if prop in props.columns else np.nan

path_rows = []
seed_rows = []
property_rows = []

for prop in selected_properties:
    y = panel[prop].to_numpy(dtype=float)
    ridge_model, z_scaler, masks, _, scores = arc.fit_ridge_probe(Z, y, split, alpha=1.0)
    direction = np.asarray(ridge_model.coef_, dtype=float)
    direction = direction / max(np.linalg.norm(direction), 1e-12)

    test_mask = masks["test"]
    candidate_idx = np.where(test_mask)[0]
    candidate_idx = candidate_idx[np.isfinite(y[candidate_idx])]
    if len(candidate_idx) == 0:
        warnings.warn(f"No test seeds for {prop}")
        continue

    qs = np.quantile(y[candidate_idx], [0.2, 0.5, 0.8])
    seed_indices = []
    for q in qs:
        nearest = candidate_idx[np.argmin(np.abs(y[candidate_idx] - q))]
        seed_indices.append(int(nearest))
    seed_indices = list(dict.fromkeys(seed_indices))[:N_SEEDS_PER_PROPERTY]

    for seed_rank, seed_idx in enumerate(seed_indices):
        z0 = Z[seed_idx]
        z_path = np.stack([z0 + alpha * direction for alpha in ALPHAS]).astype(np.float32)
        decoded = arc.decode_latents(model, z_path, bundle["id2tok"], device, batch_size=128)
        pred_values = ridge_model.predict(z_scaler.transform(z_path))
        actual_values = []

        for step, (alpha, rec, pred_value) in enumerate(zip(ALPHAS, decoded, pred_values)):
            actual = actual_property(rec["canonical_smiles"], prop) if rec["valid"] else np.nan
            actual_values.append(actual)
            row = {
                "property": prop,
                "seed_rank": seed_rank,
                "seed_index": int(seed_idx),
                "step": step,
                "alpha": float(alpha),
                "seed_property_value": float(y[seed_idx]),
                "predicted_ridge_value": float(pred_value),
                "actual_rdkit_property_value": actual,
                **decoded_confound_values(rec["selfies"]),
                **rec,
            }
            path_rows.append(row)

        actual_arr = np.asarray(actual_values, dtype=float)
        finite = np.isfinite(actual_arr)
        if finite.sum() >= 3:
            rho = pd.Series(ALPHAS[finite]).corr(pd.Series(actual_arr[finite]), method="spearman")
            slope = float(np.polyfit(ALPHAS[finite], actual_arr[finite], 1)[0])
            expected_sign = np.sign(ridge_model.coef_.dot(direction))
            diffs = np.diff(actual_arr[finite])
            violations = int(np.sum(diffs * expected_sign < -1e-9))
        else:
            rho, slope, violations = np.nan, np.nan, np.nan

        seed_rows.append({
            "property": prop,
            "seed_rank": seed_rank,
            "seed_index": int(seed_idx),
            "valid_fraction": float(np.mean([r["valid"] for r in decoded])),
            "unique_valid_along_path": int(pd.Series([r["canonical_smiles"] for r in decoded if r["valid"]]).nunique()),
            "spearman_step_vs_actual": float(rho) if pd.notna(rho) else np.nan,
            "slope_actual_vs_alpha": slope,
            "monotonicity_violation_count": violations,
        })

path_df = pd.DataFrame(path_rows)
seed_df = pd.DataFrame(seed_rows)
path_df.to_csv(TABLES_DIR / "traversal_decoded_paths.csv", index=False)
seed_df.to_csv(TABLES_DIR / "traversal_seed_summary.csv", index=False)

if len(seed_df):
    property_summary = (
        seed_df.groupby("property", as_index=False)
        .agg(
            valid_fraction=("valid_fraction", "mean"),
            median_spearman=("spearman_step_vs_actual", "median"),
            median_slope=("slope_actual_vs_alpha", "median"),
            mean_monotonicity_violations=("monotonicity_violation_count", "mean"),
            n_seeds=("seed_index", "count"),
        )
    )
else:
    property_summary = pd.DataFrame()
property_summary.to_csv(TABLES_DIR / "traversal_property_summary.csv", index=False)
display(property_summary)


## Plots And Representative Molecule Strips


In [ ]:
strip_rows = []
for prop, prop_df in path_df.groupby("property"):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for seed_rank, seed_df_plot in prop_df.groupby("seed_rank"):
        seed_df_plot = seed_df_plot.sort_values("alpha")
        axes[0].plot(seed_df_plot["alpha"], seed_df_plot["actual_rdkit_property_value"], marker="o", label=f"seed {seed_rank}")
        axes[1].plot(seed_df_plot["alpha"], seed_df_plot["predicted_ridge_value"], marker="o", label=f"seed {seed_rank}")
        axes[2].plot(seed_df_plot["alpha"], seed_df_plot["decoded_selfies_len_tokens"], marker="o", label=f"seed {seed_rank}")
    axes[0].set_title(f"{prop}: decoded RDKit property trajectory")
    axes[1].set_title(f"{prop}: predicted ridge trajectory")
    axes[2].set_title(f"{prop}: SELFIES length trajectory")
    for ax in axes:
        ax.set_xlabel("alpha")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel(prop)
    axes[1].set_ylabel("predicted value")
    axes[2].set_ylabel("SELFIES tokens")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"traversal_{prop}_trajectories.png", dpi=180)
    plt.show()

    best = (
        seed_df[seed_df["property"] == prop]
        .sort_values(["valid_fraction", "spearman_step_vs_actual"], ascending=False)
        .head(1)
    )
    if len(best):
        seed_rank = int(best.iloc[0]["seed_rank"])
        strip_df = prop_df[prop_df["seed_rank"] == seed_rank].sort_values("alpha")
        valid_strip = strip_df[strip_df["valid"]].copy()
        if len(valid_strip):
            legends = [f"a={row.alpha:.1f}\n{prop}={row.actual_rdkit_property_value:.2g}" for row in valid_strip.itertuples()]
            img = arc.molecule_grid(valid_strip["canonical_smiles"].tolist(), legends=legends, mols_per_row=min(8, len(valid_strip)))
            out_path = PLOTS_DIR / f"traversal_{prop}_representative_strip.png"
            img.save(str(out_path))
            strip_rows.append({"property": prop, "seed_rank": seed_rank, "path": str(out_path)})

pd.DataFrame(strip_rows).to_csv(TABLES_DIR / "representative_molecule_strips.csv", index=False)
summary = {
    "checkpoint": str(ckpt_path),
    "selected_properties": selected_properties,
    "excluded_as_confounded_or_controls": excluded_as_confounded,
    "n_path_rows": int(len(path_df)),
    "n_seed_rows": int(len(seed_df)),
    "summary_table": str(TABLES_DIR / "traversal_property_summary.csv"),
}
arc.write_json(OUTPUT_DIR / "summary_traversal.json", summary)
display(pd.DataFrame([summary]))
